# 03 · Política de risco aceitável

**Pergunta:** a partir de que probabilidade de o empregador deixar de existir o consignado deixa
de valer a pena?

**O que se perde quando o empregador sai.** O desconto em folha para. Isso não é calote
automático: o funcionário pode continuar pagando, levar a dívida para o novo empregador ou acionar
a garantia do FGTS. O que se perde é o canal de desconto, que é a garantia que torna o
consignado mais barato que o crédito pessoal. A política limita a chance de perder esse canal
durante o prazo.

**Forma da regra:** conceder um prazo de *t* meses se P(empregador sair até *t*) ≤ L. O prazo
máximo é o maior *t* que respeita o limite. Empresas mais arriscadas recebem prazos mais curtos em
vez de uma recusa seca.

**Alternativa descartada:** um limite de risco *anual*. Como o risco anual de cada empresa é
quase constante, ele vira "aprova todos os prazos ou nenhum" e não produz um prazo máximo útil.

Este notebook simula a política no conjunto de teste (empresas que o modelo nunca viu), usando o
modelo do agent (`artifacts/model`).

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from consig_risk.modeling import data as mdata
from consig_risk.modeling.artifact import RiskModel
from consig_risk.modeling.metrics import horizon_labels
from consig_risk.risk.policy import RiskPolicy

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")

modelo = RiskModel.load(Path("artifacts/model"))
teste = mdata.load_cohort(Path("data/processed/cohort.parquet"), mdata.TEST)
PRAZOS = (6, 12, 18, 24, 30, 36, 48, 60)
risco = 1 - modelo.survival(teste, PRAZOS)
print(f"{len(teste):,} empresas do conjunto de teste")

954,274 empresas do conjunto de teste


## 1. Como o risco se distribui entre os empregadores

In [2]:
dist = pd.DataFrame(
    100 * np.percentile(risco, [10, 25, 50, 75, 90], axis=0),
    index=["10% menos arriscadas", "25%", "mediana", "75%", "10% mais arriscadas"],
    columns=[f"{p} meses" for p in PRAZOS],
)
print("risco previsto de o empregador sair até o fim do prazo (%)")
display(dist)

risco previsto de o empregador sair até o fim do prazo (%)


,6 meses,12 meses,18 meses,24 meses,30 meses,36 meses,48 meses,60 meses
10% menos arriscadas,0.6,1.2,1.8,2.5,3.5,4.5,6.4,8.2
25%,1.4,2.7,4.3,5.8,8.2,10.5,14.6,18.3
mediana,2.6,5.1,7.8,10.4,14.3,18.0,24.6,30.3
75%,4.3,8.5,12.9,17.1,22.7,27.9,37.0,44.6
10% mais arriscadas,7.0,13.5,20.7,27.4,35.9,43.4,55.7,65.1


O risco cresce com o prazo para todas as empresas, mas em ritmos muito diferentes: em 24 meses, a
mediana é cerca de 10%, e os 10% mais arriscados passam de 27%.

## 2. O que cada limite significaria

O limite é aplicado ao prazo de 24 meses, que tem desfecho observado para todas as empresas do
teste. Para cada limite candidato: quantos empregadores seriam aprovados, quantos dos aprovados de
fato saíram e quantas das empresas que saíram teriam sido barradas.

In [3]:
def simula(prazo: int) -> pd.DataFrame:
    j = PRAZOS.index(prazo)
    y = horizon_labels(teste, prazo)
    linhas = []
    for limite in (0.025, 0.05, 0.075, 0.10, 0.15, 0.20):
        aprovada = risco[:, j] <= limite
        linhas.append(
            {
                "limite": f"{100 * limite:.1f}%",
                "empregadores aprovados (%)": 100 * aprovada.mean(),
                f"saíram em {prazo}m, entre aprovados (%)": 100 * y[aprovada].mean(),
                f"saíram em {prazo}m, entre recusados (%)": 100 * y[~aprovada].mean(),
                "das que saíram, barradas (%)": 100 * y[~aprovada].sum() / y.sum(),
            }
        )
    return pd.DataFrame(linhas).set_index("limite")


print(f"taxa média de saída em 24 meses: {100 * horizon_labels(teste, 24).mean():.1f}%")
display(simula(24))
print(f"\ntaxa média de saída em 12 meses: {100 * horizon_labels(teste, 12).mean():.1f}%")
display(simula(12))

taxa média de saída em 24 meses: 12.9%


,empregadores aprovados (%),"saíram em 24m, entre aprovados (%)","saíram em 24m, entre recusados (%)","das que saíram, barradas (%)"
limite,,,,
2.5%,10.2,1.0,14.3,99.2
5.0%,21.0,2.3,15.8,96.3
7.5%,34.3,3.8,17.7,89.9
10.0%,48.2,5.2,20.1,80.6
15.0%,69.2,7.4,25.5,60.6
20.0%,81.1,8.8,30.7,44.9



taxa média de saída em 12 meses: 6.6%


,empregadores aprovados (%),"saíram em 12m, entre aprovados (%)","saíram em 12m, entre recusados (%)","das que saíram, barradas (%)"
limite,,,,
2.5%,22.5,1.1,8.2,96.1
5.0%,49.3,2.6,10.6,80.9
7.5%,69.5,3.6,13.5,62.2
10.0%,81.2,4.3,16.5,47.0
15.0%,92.6,5.2,23.8,26.5
20.0%,96.9,5.8,33.3,15.7


**Escolha: L = 10%.**

- Aprova cerca de metade dos empregadores para 24 meses (e cerca de 80% para 12 meses).
- Entre os aprovados, a saída real em 24 meses fica em torno de 5%, cerca de 2,5 vezes abaixo da
  média do público. Entre os recusados, fica perto de 20%.
- Barra cerca de 80% das empresas que de fato saíram.
- "1 chance em 10 de perder o desconto em folha durante o prazo" é fácil de explicar ao
  analista e ao comitê de crédito.

Limites menores (5%) cortam demais a oferta comercial; limites maiores (15–20%) deixam passar
metade das empresas que fecham.

## 3. Prazo máximo resultante

In [4]:
politica = RiskPolicy(_env_file=None)
idx = [PRAZOS.index(p) for p in politica.prazos()]
aceito = risco[:, idx] <= politica.limite_risco
maximo = np.where(aceito.any(axis=1), np.array(politica.prazos())[aceito.sum(axis=1) - 1], 0)
# A curva de risco é não decrescente, então os prazos aceitos são sempre os primeiros da grade.
assert (aceito.sum(axis=1) == aceito.cumprod(axis=1).sum(axis=1)).all()
distribuicao = (
    pd.Series(maximo)
    .map(lambda m: "não conceder" if m == 0 else f"até {m} meses")
    .value_counts(normalize=True)
    .mul(100)
    .reindex(["não conceder", *[f"até {p} meses" for p in politica.prazos()]])
    .fillna(0)
)
print(
    f"política: limite {politica.limite_risco:.0%}, prazo máximo {politica.prazo_maximo_meses} meses"
)
display(distribuicao.to_frame("% das empresas"))

política: limite 10%, prazo máximo 36 meses


,% das empresas
não conceder,3.6
até 6 meses,15.2
até 12 meses,18.1
até 18 meses,14.9
até 24 meses,16.0
até 30 meses,8.6
até 36 meses,23.6


**Prazo máximo de 36 meses:** é o horizonte em que o modelo foi validado com desfecho observado.
A política nunca aprova com base em risco extrapolado. O mercado vai até cerca de 48 meses no
consignado privado; estender o teto para 48 é uma decisão de negócio possível, com o aviso de
extrapolação que o agent já dá (`RISK_PRAZO_MAXIMO_MESES`).

## 4. Regras antes do modelo e avisos

| Situação | Resultado | Por quê |
|---|---|---|
| CNPJ inválido ou inexistente | O agent diz que não pode analisar | Não há dado; nada é inventado |
| Empresa não ativa | Não conceder | Não há empregador ativo |
| Situação especial (recuperação judicial, falência) | Não conceder | Risco jurídico fora do que o modelo mede |
| MEI, administração pública, pessoa física, organismo internacional | Fora do escopo | O modelo não foi treinado nesses casos |
| Empresa com menos de 3 anos | Aviso | O modelo separa pouco empresas jovens (AUC 0,63–0,66) |
| Valores fora da faixa do treino | Aviso | As árvores tratam como o maior valor visto |
| Prazo pedido acima de 36 meses | Não conceder, com o risco extrapolado mostrado | Fora do horizonte validado |

## 5. Evolução natural

Ponderar o risco pelo saldo devedor e pela perda esperada. O saldo cai ao longo do prazo, então
perder o canal no mês 30 custa menos do que no mês 3: perda esperada ≈ Σ P(sair no mês m) × saldo
no mês m × perda dado o evento. Isso exige taxa de juros, tabela de amortização e uma estimativa
de recuperação (portabilidade, FGTS) que não estão nos dados públicos. Fica como evolução.